In [ ]:
#Importaciones

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

print("Librerías cargadas.")


In [ ]:
# Data 
df = pd.read_csv("../data/tickets_train.csv")
df.head()

In [ ]:
#Variables predictorias 
text = df["ticket_text"].astype(str)

extra_features = df[[
    "sentiment_label",
    "is_phishing",
    "has_pii",
    "project_age_months",
    "tickets_corrective_last_30d",
    "tickets_evolutionary_last_30d",
    "tickets_total_client"
]]

y = df["churn_risk"]


In [ ]:
#Vectorización
vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")

X_text = vectorizer.transform(text)
X = np.hstack((X_text.toarray(), extra_features.values))

X.shape


In [ ]:
#Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
#Entrenar modelo de churn
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    random_state=42
)

model.fit(X_train, y_train)
print("Modelo entrenado.")


In [ ]:
#Evaluación
y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

plt.figure(figsize=(6,4))
sns.scatterplot(x=y_test, y=y_pred)
plt.title("Real vs Predicho — Churn")
plt.xlabel("Churn Real")
plt.ylabel("Churn Predicho")
plt.show()


In [ ]:
#Importancia de features
feature_names = (
    [f"tfidf_{i}" for i in range(X_text.shape[1])]
    + list(extra_features.columns)
)

importances = model.feature_importances_

drivers_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
)

drivers_df.head(20)


In [ ]:
#Drivers importantes
drivers_df.to_csv("../models/churn_drivers.csv", index=False)
print("Drivers guardados.")


In [ ]:
#Guardar modelo

joblib.dump(model, "../models/model_churn.pkl")

print("Modelo de churn guardado.")
